# Class Imbalance 실험
- 1순위 중분류 예측에서 소수 분류 예측 성능을 점검함.
- 다항 로지스틱 기반으로 class weight와 sampling 전략을 비교함.
- ROL 보정은 별도 손실함수 설계가 필요하므로 후속 실험으로 분리함.

## 분석 환경 및 데이터 경로 설정
- 분석에 필요한 패키지와 경로를 설정함.
- 01번 노트북에서 생성한 순위모형 기초 테이블을 사용함.

In [ ]:
import pathlib
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    log_loss,
    f1_score,
    recall_score,
    precision_recall_fscore_support,
    balanced_accuracy_score,
)
from sklearn.utils.class_weight import compute_class_weight

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

BASE_PATH = pathlib.Path().resolve()

PROJECT_PATH = None
for path in [BASE_PATH, *BASE_PATH.parents]:
    if (path / "notebooks" / "preference" / "data").exists():
        PROJECT_PATH = path
        break

if PROJECT_PATH is None:
    raise FileNotFoundError("???? ??? ?? ?????.")

PREFERENCE_PATH = PROJECT_PATH / "notebooks" / "preference"
PROCESSED_PATH = PREFERENCE_PATH / "data" / "processed" / "satisfaction"

RANK_BASE_PATH = PROCESSED_PATH / "ml_preference_satisfaction_rank_base.csv"
MAPPING_PATH = PROCESSED_PATH / "ml_activity_category_mapping.csv"

print("PROJECT_PATH:", PROJECT_PATH)
print("RANK_BASE_PATH 존재:", RANK_BASE_PATH.exists())
print("MAPPING_PATH 존재:", MAPPING_PATH.exists())

## 데이터 불러오기 및 라벨 부여
- 만족 여가활동 순위 테이블을 불러옴.
- 성별, 연령대, 시도, 지역규모 라벨을 생성함.
- 학습 대상 중분류 목록을 확정함.

In [ ]:
rank_base = pd.read_csv(RANK_BASE_PATH, encoding="utf-8-sig", low_memory=False)
activity_mapping = pd.read_csv(MAPPING_PATH, encoding="utf-8-sig")

rank_cols = [
    "만족_유효중분류_1순위",
    "만족_유효중분류_2순위",
    "만족_유효중분류_3순위",
]

target_col = "만족_유효중분류_1순위"

valid_categories = (
    activity_mapping
    .loc[activity_mapping["학습타깃사용여부"], "중분류"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

sex_map = {
    1: "남성",
    2: "여성",
}

age_map = {
    1: "15-19세",
    2: "20대",
    3: "30대",
    4: "40대",
    5: "50대",
    6: "60대",
    7: "70세 이상",
}

sido_map = {
    1: "서울",
    2: "부산",
    3: "대구",
    4: "인천",
    5: "광주",
    6: "대전",
    7: "울산",
    8: "세종",
    9: "경기",
    10: "강원",
    11: "충북",
    12: "충남",
    13: "전북",
    14: "전남",
    15: "경북",
    16: "경남",
    17: "제주",
}

region_size_map = {
    1: "대도시",
    2: "중소도시",
    3: "읍면지역",
}

model_df = rank_base.copy()
model_df["성별_라벨"] = model_df["성별"].map(sex_map)
model_df["연령대"] = model_df["연령"].map(age_map)
model_df["시도"] = model_df["17개 시도"].map(sido_map)
model_df["지역규모_라벨"] = model_df["지역규모"].map(region_size_map)
model_df["조사년도_라벨"] = model_df["조사년도"].astype(str)

model_feature_cols = [
    "성별_라벨",
    "연령대",
    "시도",
    "지역규모_라벨",
    "조사년도_라벨",
]

model_df = model_df[model_df[target_col].isin(valid_categories)].copy()

print("데이터 구조:", model_df.shape)
print("학습 대상 중분류:", valid_categories)
print("1순위 결측:", model_df[target_col].isna().sum())
print("입력 변수 결측")
print(model_df[model_feature_cols].isna().sum())

display(model_df.head())

## 1순위 중분류 불균형 확인
- 1순위 중분류 분포를 확인함.
- 표본 수가 작은 하위 5개 중분류를 소수 분류로 지정함.

In [ ]:
target_distribution = (
    model_df[target_col]
    .value_counts()
    .reindex(valid_categories)
    .reset_index()
)
target_distribution.columns = ["중분류", "응답자수"]
target_distribution["비율"] = target_distribution["응답자수"] / len(model_df)

display(target_distribution.sort_values("응답자수", ascending=False))

minority_categories = (
    target_distribution
    .sort_values("응답자수")
    .head(5)["중분류"]
    .tolist()
)

print("소수 분류:", minority_categories)

## Train / Valid / Test 분리
- 70:15:15 비율로 데이터를 분리함.
- 조사년도와 1순위 중분류 분포가 유지되도록 stratify를 적용함.

In [ ]:
split_key = (
    model_df["조사년도_라벨"]
    + "_"
    + model_df[target_col]
)

train_valid_idx, test_idx = train_test_split(
    model_df.index,
    test_size=0.15,
    random_state=42,
    stratify=split_key,
)

train_valid_df = model_df.loc[train_valid_idx].copy()
train_valid_key = (
    train_valid_df["조사년도_라벨"]
    + "_"
    + train_valid_df[target_col]
)

train_idx, valid_idx = train_test_split(
    train_valid_df.index,
    test_size=0.15 / 0.85,
    random_state=42,
    stratify=train_valid_key,
)

train_df = model_df.loc[train_idx].copy()
valid_df = model_df.loc[valid_idx].copy()
test_df = model_df.loc[test_idx].copy()

split_summary = pd.DataFrame({
    "데이터": ["train", "valid", "test"],
    "응답자수": [len(train_df), len(valid_df), len(test_df)],
    "비율": [len(train_df) / len(model_df), len(valid_df) / len(model_df), len(test_df) / len(model_df)],
})

display(split_summary)

split_target_distribution = pd.crosstab(
    pd.concat([
        train_df.assign(split="train"),
        valid_df.assign(split="valid"),
        test_df.assign(split="test"),
    ])[target_col],
    pd.concat([
        train_df.assign(split="train"),
        valid_df.assign(split="valid"),
        test_df.assign(split="test"),
    ])["split"],
    normalize="columns",
)

display(split_target_distribution)

## 모델 입력 배열 생성
- 범주형 입력변수를 더미 변수로 변환함.
- 최종가중치를 평균 1로 정규화해 학습 가중치로 사용함.
- 1~3순위 중분류를 평가용 배열로 변환함.

In [ ]:
feature_df = pd.get_dummies(
    model_df[model_feature_cols],
    drop_first=True,
    dtype=float,
)

feature_columns = feature_df.columns.tolist()

category_to_idx = {
    category: idx
    for idx, category in enumerate(valid_categories)
}

idx_to_category = {
    idx: category
    for category, idx in category_to_idx.items()
}


def make_rank_array(data):
    y_rank = np.full((len(data), 3), -1, dtype=int)
    
    for rank, col in enumerate(rank_cols):
        y_rank[:, rank] = (
            data[col]
            .map(category_to_idx)
            .fillna(-1)
            .astype(int)
            .to_numpy()
        )
    
    return y_rank


def normalized_weight(data):
    sample_weight = data["최종가중치"].to_numpy(dtype=float)
    return sample_weight / np.nanmean(sample_weight)

X_train = feature_df.loc[train_df.index].to_numpy(dtype=float)
X_valid = feature_df.loc[valid_df.index].to_numpy(dtype=float)
X_test = feature_df.loc[test_df.index].to_numpy(dtype=float)

y_train = train_df[target_col].to_numpy()
y_valid = valid_df[target_col].to_numpy()
y_test = test_df[target_col].to_numpy()

y_rank_train = make_rank_array(train_df)
y_rank_valid = make_rank_array(valid_df)
y_rank_test = make_rank_array(test_df)

weight_train = normalized_weight(train_df)
weight_valid = normalized_weight(valid_df)
weight_test = normalized_weight(test_df)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("X_test:", X_test.shape)
print("입력 더미 변수 수:", len(feature_columns))

## 평가 함수 정의
- 전체 성능 지표와 소수 분류 성능 지표를 함께 산출함.
- Top1, Top3, MRR, NDCG@3, LogLoss, Macro F1, Balanced Accuracy를 계산함.

In [ ]:
def ndcg_at_3(prob, y_rank):
    pred_order = np.argsort(-prob, axis=1)[:, :3]
    ndcg_list = []
    
    for i in range(len(y_rank)):
        relevance = {}
        
        for rank in range(3):
            if y_rank[i, rank] >= 0:
                relevance[y_rank[i, rank]] = 3 - rank
        
        if len(relevance) == 0:
            continue
        
        dcg = 0.0
        
        for position, category_idx in enumerate(pred_order[i], start=1):
            rel = relevance.get(category_idx, 0)
            dcg += (2 ** rel - 1) / np.log2(position + 1)
        
        ideal_relevance = sorted(relevance.values(), reverse=True)[:3]
        idcg = sum(
            (2 ** rel - 1) / np.log2(position + 1)
            for position, rel in enumerate(ideal_relevance, start=1)
        )
        
        ndcg_list.append(dcg / idcg if idcg > 0 else np.nan)
    
    return np.nanmean(ndcg_list)


def align_prob(model, X):
    raw_prob = model.predict_proba(X)
    prob = np.zeros((X.shape[0], len(valid_categories)))
    class_to_col = {
        category: idx
        for idx, category in enumerate(model.classes_)
    }
    
    for j, category in enumerate(valid_categories):
        if category in class_to_col:
            prob[:, j] = raw_prob[:, class_to_col[category]]
    
    row_sum = prob.sum(axis=1, keepdims=True)
    prob = np.divide(
        prob,
        row_sum,
        out=np.zeros_like(prob),
        where=row_sum > 0,
    )
    
    return prob


def evaluate_prob_model(model_name, split_name, prob, y_true, y_rank):
    pred_order = np.argsort(-prob, axis=1)
    pred_idx = pred_order[:, 0]
    pred_label = np.array([idx_to_category[idx] for idx in pred_idx])
    rank1_idx = np.array([category_to_idx[label] for label in y_true])
    
    top1_accuracy = np.mean(pred_label == y_true)
    top3_hit_rate = np.mean([
        rank1_idx[i] in pred_order[i, :3]
        for i in range(len(rank1_idx))
    ])
    mrr = np.mean([
        1 / (np.where(pred_order[i] == rank1_idx[i])[0][0] + 1)
        for i in range(len(rank1_idx))
    ])
    ndcg = ndcg_at_3(prob, y_rank)
    rank1_logloss = log_loss(y_true, prob, labels=valid_categories)
    macro_f1 = f1_score(y_true, pred_label, labels=valid_categories, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_true, pred_label, labels=valid_categories, average="weighted", zero_division=0)
    balanced_acc = balanced_accuracy_score(y_true, pred_label)
    minority_recall = recall_score(
        y_true,
        pred_label,
        labels=minority_categories,
        average="macro",
        zero_division=0,
    )
    minority_mask = pd.Series(y_true).isin(minority_categories).to_numpy()
    minority_top3_hit = np.mean([
        rank1_idx[i] in pred_order[i, :3]
        for i in np.where(minority_mask)[0]
    ]) if minority_mask.sum() > 0 else np.nan
    
    return {
        "모델": model_name,
        "데이터": split_name,
        "Top1_Accuracy": top1_accuracy,
        "Top3_HitRate": top3_hit_rate,
        "MRR": mrr,
        "NDCG@3": ndcg,
        "Rank1_LogLoss": rank1_logloss,
        "Macro_F1": macro_f1,
        "Weighted_F1": weighted_f1,
        "Balanced_Accuracy": balanced_acc,
        "소수분류_Recall": minority_recall,
        "소수분류_Top3_HitRate": minority_top3_hit,
    }


def make_category_metric(model_name, split_name, prob, y_true):
    pred_idx = np.argmax(prob, axis=1)
    pred_label = np.array([idx_to_category[idx] for idx in pred_idx])
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        pred_label,
        labels=valid_categories,
        zero_division=0,
    )
    pred_count = pd.Series(pred_label).value_counts().reindex(valid_categories, fill_value=0).to_numpy()
    
    return pd.DataFrame({
        "모델": model_name,
        "데이터": split_name,
        "중분류": valid_categories,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "실제건수": support,
        "예측건수": pred_count,
        "소수분류여부": [category in minority_categories for category in valid_categories],
    })

## 실험 모델 학습
- 기본 다항 로지스틱을 학습함.
- class_weight balanced 모델을 학습함.
- class_weight를 완만하게 적용한 sqrt balanced 모델을 학습함.
- train 데이터에서 소수 분류를 복원추출한 oversampling 모델을 학습함.

In [ ]:
def fit_mnl_model(model_name, X, y, sample_weight=None, class_weight=None):
    model = LogisticRegression(
        solver="lbfgs",
        max_iter=1000,
        C=1.0,
        class_weight=class_weight,
    )
    model.fit(X, y, sample_weight=sample_weight)
    print(model_name, "수렴 반복 횟수:", model.n_iter_[0])
    return model

# 1. 기본 모델
mnl_base = fit_mnl_model(
    "기본",
    X_train,
    y_train,
    sample_weight=weight_train,
)

# 2. class_weight balanced 모델
mnl_class_weight = fit_mnl_model(
    "class_weight_balanced",
    X_train,
    y_train,
    sample_weight=weight_train,
    class_weight="balanced",
)

# 3. sqrt balanced 모델
class_weight_values = compute_class_weight(
    class_weight="balanced",
    classes=np.array(valid_categories),
    y=y_train,
)

sqrt_class_weight = {
    category: np.sqrt(weight)
    for category, weight in zip(valid_categories, class_weight_values)
}

sqrt_sample_weight = weight_train * pd.Series(y_train).map(sqrt_class_weight).to_numpy()
sqrt_sample_weight = sqrt_sample_weight / np.nanmean(sqrt_sample_weight)

mnl_sqrt_weight = fit_mnl_model(
    "sqrt_class_weight",
    X_train,
    y_train,
    sample_weight=sqrt_sample_weight,
)

# 4. random oversampling 모델
train_count = train_df[target_col].value_counts()
over_target_count = train_count.max()

sampled_index_list = []

for category, group in train_df.groupby(target_col):
    sampled_index = group.sample(
        n=over_target_count,
        replace=len(group) < over_target_count,
        random_state=42,
    ).index
    sampled_index_list.append(sampled_index.to_numpy())

over_idx = pd.Index(np.concatenate(sampled_index_list))
over_train_df = model_df.loc[over_idx].copy()

X_train_over = feature_df.loc[over_idx].to_numpy(dtype=float)
y_train_over = over_train_df[target_col].to_numpy()
weight_train_over = normalized_weight(over_train_df)

mnl_oversampling = fit_mnl_model(
    "random_oversampling",
    X_train_over,
    y_train_over,
    sample_weight=weight_train_over,
)

print()
print("oversampling 전 train 구조:", train_df.shape)
print("oversampling 후 train 구조:", over_train_df.shape)
over_distribution = (
    over_train_df[target_col]
    .value_counts()
    .reindex(valid_categories)
    .rename_axis("중분류")
    .reset_index(name="응답자수")
)

display(over_distribution)

## 실험 성능 비교
- valid/test 기준으로 모델별 전체 성능과 소수 분류 성능을 비교함.
- prior 기준모형을 함께 제시함.

In [ ]:
train_prior = train_df[target_col].value_counts(normalize=True)
prior_prob = np.array([
    train_prior.get(category, 0)
    for category in valid_categories
])
prior_prob = prior_prob / prior_prob.sum()

experiment_models = {
    "prior": None,
    "기본": mnl_base,
    "class_weight_balanced": mnl_class_weight,
    "sqrt_class_weight": mnl_sqrt_weight,
    "random_oversampling": mnl_oversampling,
}

split_data = {
    "train": (X_train, y_train, y_rank_train),
    "valid": (X_valid, y_valid, y_rank_valid),
    "test": (X_test, y_test, y_rank_test),
}

performance_rows = []
category_metric_list = []

for model_name, model in experiment_models.items():
    for split_name, (X_split, y_split, y_rank_split) in split_data.items():
        if model_name == "prior":
            prob = np.tile(prior_prob, (len(y_split), 1))
        else:
            prob = align_prob(model, X_split)
        
        performance_rows.append(
            evaluate_prob_model(model_name, split_name, prob, y_split, y_rank_split)
        )
        
        if split_name in ["valid", "test"]:
            category_metric_list.append(
                make_category_metric(model_name, split_name, prob, y_split)
            )

imbalance_performance = pd.DataFrame(performance_rows).round(4)
category_metric_table = pd.concat(category_metric_list, ignore_index=True).round(4)

display(imbalance_performance)

print("valid 성능 비교")
display(imbalance_performance[imbalance_performance["데이터"] == "valid"].sort_values("Macro_F1", ascending=False))

print("test 성능 비교")
display(imbalance_performance[imbalance_performance["데이터"] == "test"].sort_values("Macro_F1", ascending=False))

## 중분류별 성능 점검
- valid/test의 중분류별 Recall, F1, 예측건수를 확인함.
- 소수 분류가 실제로 예측되는지 점검함.

In [ ]:
print("valid 중분류별 성능")
display(
    category_metric_table[category_metric_table["데이터"] == "valid"]
    .sort_values(["중분류", "모델"])
)

print("test 중분류별 성능")
display(
    category_metric_table[category_metric_table["데이터"] == "test"]
    .sort_values(["중분류", "모델"])
)

print("소수 분류 test 성능")
display(
    category_metric_table[
        (category_metric_table["데이터"] == "test")
        & (category_metric_table["소수분류여부"])
    ].sort_values(["중분류", "모델"])
)

## 실험 결과 정리
- class imbalance 보정 전후 성능표를 생성함.
- 전체 성능과 소수 분류 성능을 함께 비교함.
- focal loss는 별도 모델 계열에서 후속 실험으로 검토함.

In [ ]:
best_valid_macro = (
    imbalance_performance[imbalance_performance["데이터"] == "valid"]
    .sort_values("Macro_F1", ascending=False)
    .head(1)
)

best_test_macro = (
    imbalance_performance[imbalance_performance["데이터"] == "test"]
    .sort_values("Macro_F1", ascending=False)
    .head(1)
)

best_valid_minority = (
    imbalance_performance[imbalance_performance["데이터"] == "valid"]
    .sort_values("소수분류_Recall", ascending=False)
    .head(1)
)

best_test_minority = (
    imbalance_performance[imbalance_performance["데이터"] == "test"]
    .sort_values("소수분류_Recall", ascending=False)
    .head(1)
)

print("valid Macro F1 최고 모델")
display(best_valid_macro)

print("test Macro F1 최고 모델")
display(best_test_macro)

print("valid 소수분류 Recall 최고 모델")
display(best_valid_minority)

print("test 소수분류 Recall 최고 모델")
display(best_test_minority)

print("저장하지 않은 임시 실험 테이블")
print("imbalance_performance:", imbalance_performance.shape)
print("category_metric_table:", category_metric_table.shape)

## Class weight 강도 조절 실험
- balanced class weight를 지수로 완화한 실험을 수행함.
- alpha가 커질수록 소수 분류 보정 강도가 커지도록 설정함.
- 전체 성능과 소수 분류 성능의 trade-off를 비교함.

In [ ]:
alpha_list = [
    0.00,
    0.10,
    0.20,
    0.25,
    0.30,
    0.40,
    0.50,
    0.60,
    0.75,
    1.00,
]

base_class_weight_values = compute_class_weight(
    class_weight="balanced",
    classes=np.array(valid_categories),
    y=y_train,
)

base_class_weight = {
    category: weight
    for category, weight in zip(valid_categories, base_class_weight_values)
}

class_weight_alpha_rows = []

for alpha in alpha_list:
    temp = {"alpha": alpha}
    
    for category in valid_categories:
        temp[category] = base_class_weight[category] ** alpha
    
    class_weight_alpha_rows.append(temp)

class_weight_alpha_table = pd.DataFrame(class_weight_alpha_rows).round(4)

display(class_weight_alpha_table)

## Alpha별 모델 학습
- alpha별 sample weight를 생성함.
- 동일한 train/valid/test 분할에서 다항 로지스틱을 반복 학습함.
- valid/test 성능을 비교함.

In [ ]:
alpha_model_dict = {}
alpha_performance_rows = []
alpha_category_metric_list = []

for alpha in alpha_list:
    adjusted_class_weight = {
        category: base_class_weight[category] ** alpha
        for category in valid_categories
    }
    
    alpha_sample_weight = weight_train * pd.Series(y_train).map(adjusted_class_weight).to_numpy()
    alpha_sample_weight = alpha_sample_weight / np.nanmean(alpha_sample_weight)
    
    model_name = f"alpha_{alpha:.2f}"
    
    model = LogisticRegression(
        solver="lbfgs",
        max_iter=1000,
        C=1.0,
    )
    
    model.fit(
        X_train,
        y_train,
        sample_weight=alpha_sample_weight,
    )
    
    alpha_model_dict[alpha] = model
    print(model_name, "수렴 반복 횟수:", model.n_iter_[0])
    
    for split_name, X_split, y_split, y_rank_split in [
        ("valid", X_valid, y_valid, y_rank_valid),
        ("test", X_test, y_test, y_rank_test),
    ]:
        prob = align_prob(model, X_split)
        alpha_performance_rows.append(
            evaluate_prob_model(model_name, split_name, prob, y_split, y_rank_split)
        )
        alpha_category_metric_list.append(
            make_category_metric(model_name, split_name, prob, y_split)
        )

alpha_performance = pd.DataFrame(alpha_performance_rows)
alpha_category_metric = pd.concat(alpha_category_metric_list, ignore_index=True)

base_reference = (
    alpha_performance[alpha_performance["모델"] == "alpha_0.00"]
    [["데이터", "Top1_Accuracy", "Macro_F1", "Rank1_LogLoss", "소수분류_Recall", "소수분류_Top3_HitRate"]]
    .rename(columns={
        "Top1_Accuracy": "기본_Top1_Accuracy",
        "Macro_F1": "기본_Macro_F1",
        "Rank1_LogLoss": "기본_Rank1_LogLoss",
        "소수분류_Recall": "기본_소수분류_Recall",
        "소수분류_Top3_HitRate": "기본_소수분류_Top3_HitRate",
    })
)

alpha_performance_compare = alpha_performance.merge(
    base_reference,
    on="데이터",
    how="left",
)

alpha_performance_compare["기본대비_Top1_변화"] = (
    alpha_performance_compare["Top1_Accuracy"]
    - alpha_performance_compare["기본_Top1_Accuracy"]
)

alpha_performance_compare["기본대비_Macro_F1_변화"] = (
    alpha_performance_compare["Macro_F1"]
    - alpha_performance_compare["기본_Macro_F1"]
)

alpha_performance_compare["기본대비_LogLoss_변화"] = (
    alpha_performance_compare["Rank1_LogLoss"]
    - alpha_performance_compare["기본_Rank1_LogLoss"]
)

alpha_performance_compare["기본대비_소수분류_Recall_변화"] = (
    alpha_performance_compare["소수분류_Recall"]
    - alpha_performance_compare["기본_소수분류_Recall"]
)

alpha_performance_compare = alpha_performance_compare.round(4)

display(alpha_performance_compare)

print("valid alpha 성능")
display(
    alpha_performance_compare[alpha_performance_compare["데이터"] == "valid"]
    .sort_values("Macro_F1", ascending=False)
)

print("test alpha 성능")
display(
    alpha_performance_compare[alpha_performance_compare["데이터"] == "test"]
    .sort_values("Macro_F1", ascending=False)
)

## 후보 Alpha 선정
- 기본 모델 대비 Top1 하락폭이 3%p 이내인 후보를 선별함.
- Macro F1과 소수 분류 Recall이 개선된 후보를 확인함.
- valid 기준 후보와 test 결과를 함께 확인함.

In [ ]:
valid_alpha_compare = alpha_performance_compare[
    alpha_performance_compare["데이터"] == "valid"
].copy()

valid_alpha_compare["후보조건"] = (
    (valid_alpha_compare["기본대비_Top1_변화"] >= -0.03)
    & (valid_alpha_compare["기본대비_Macro_F1_변화"] > 0)
    & (valid_alpha_compare["기본대비_소수분류_Recall_변화"] > 0)
    & (valid_alpha_compare["기본대비_LogLoss_변화"] <= 0.20)
)

candidate_alpha_table = (
    valid_alpha_compare[valid_alpha_compare["후보조건"]]
    .sort_values(["Macro_F1", "소수분류_Recall"], ascending=False)
)

print("후보 alpha")
display(candidate_alpha_table)

if len(candidate_alpha_table) > 0:
    selected_alpha_name = candidate_alpha_table.iloc[0]["모델"]
else:
    selected_alpha_name = (
        valid_alpha_compare
        .sort_values("Macro_F1", ascending=False)
        .iloc[0]["모델"]
    )

selected_alpha = float(selected_alpha_name.replace("alpha_", ""))

selected_alpha_performance = alpha_performance_compare[
    alpha_performance_compare["모델"] == selected_alpha_name
].copy()

print("선택 alpha:", selected_alpha)
display(selected_alpha_performance)

## 선택 Alpha 중분류별 성능 확인
- 선택 alpha 모델의 중분류별 성능을 확인함.
- 기본 모델과 선택 alpha 모델의 test 성능을 비교함.

In [ ]:
base_vs_selected_category = alpha_category_metric[
    (alpha_category_metric["데이터"] == "test")
    & (alpha_category_metric["모델"].isin(["alpha_0.00", selected_alpha_name]))
].copy()

base_vs_selected_category = base_vs_selected_category.sort_values([
    "중분류",
    "모델",
]).round(4)

display(base_vs_selected_category)

selected_summary = pd.DataFrame({
    "선택_alpha": [selected_alpha],
    "선택_모델명": [selected_alpha_name],
    "valid_후보조건_충족": [len(candidate_alpha_table) > 0],
})

display(selected_summary)